IMPORT LIBRARIES

In [19]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split , StratifiedKFold
from sklearn.preprocessing import StandardScaler , OneHotEncoder , LabelEncoder , MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

IMPORT AND VIEW DATASET

In [20]:
df = pd.read_csv('india_fraud_transactions_6500_messy.csv')
df

,transaction_id,amount,channel,transaction_type,merchant,city,device_type,bank,failed_attempts,transaction_velocity,international_flag,account_age_years,location_type,fraud
0,781453,2811.68,POS,cash withdrawal,Razorpay,Bengaluru,Android,SBI,12,32.74,0,3,domestic,Not Fraud
1,496922,3682.8,UPI,Bill Payment,Jio,Hyderabad,Mac,Kotak,2,33.17,1,1,Domestic,Not Fraud
2,279451,1485.6,POS,Bill Payment,Myntra,Indore,android,PNB,9,43.77,0,3,Domestic,Not Fraud
3,429963,3327.08,UPI,Bill Payment,BigBasket,Surat,android,HDFC,4,39.36,1,3,International,Not Fraud
4,724834,2537.69,Card,Purchase,Swiggy,Chandigarh,iPhone,HDFC,8,31.79,0,0,domestic,Not Fraud
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6495,745944,1002.5,Wallet,Purchase,Swiggy,Kochi,iPhone,Kotak,5,39.79,1,1,Domestic,Not Fraud
6496,376832,697.05,ATM,Transfer,BigBasket,Bhopal,iPhone,SBI,2,40.47,1,2,Domestic,Not Fraud
6497,689173,3136.85,UPI,Cash Withdrawal,Ajio,Mumbai,Windows,ICICI,12,58.94,1,2,domestic,Not Fraud
6498,461251,3439.67,NetBanking,Transfer,SBI,Ahmedabad,Windows,Union Bank,15,14.36,0,0,International,Not Fraud


In [21]:
df['amount'] = (
    df['amount'].str.replace('₹' , '' , regex=False).str.replace(',' , '' , regex = False).str.strip().astype(float)
)
df['fraud'] = df['fraud'].str.strip().str.lower()
df['fraud'] = df['fraud'].replace({
    "fraud": 1,
    "yes": 1,
    "no": 0,
    "not fraud": 0
})
df["fraud"] = df["fraud"].astype(int)

EDA STEPS WE WILL LOOK DATA

In [22]:
df.describe()

,transaction_id,amount,failed_attempts,transaction_velocity,international_flag,account_age_years,fraud
count,6500.000000,6372.000000,6500.000000,6500.000000,6500.000000,6500.000000,6500.000000
mean,548754.598615,3735.224342,8.073077,29.878568,0.487231,1.516308,0.083385
std,259593.195932,5178.804261,4.300891,17.082981,0.499875,1.116762,0.276484
min,100126.000000,85.490000,1.000000,0.110000,0.000000,0.000000,0.000000
25%,323612.000000,1125.352500,4.000000,15.305000,0.000000,1.000000,0.000000
50%,548497.500000,2192.525000,8.000000,29.925000,0.000000,2.000000,0.000000
75%,772415.250000,4388.335000,12.000000,44.650000,1.000000,3.000000,0.000000
max,999834.000000,127637.280000,15.000000,59.990000,1.000000,3.000000,1.000000


In [23]:
df.isnull().sum()

transaction_id            0
amount                  128
channel                   6
transaction_type         16
merchant                 17
city                     17
device_type               7
bank                     10
failed_attempts           0
transaction_velocity      0
international_flag        0
account_age_years         0
location_type             3
fraud                     0
dtype: int64

In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6500 entries, 0 to 6499
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   transaction_id        6500 non-null   int64  
 1   amount                6372 non-null   float64
 2   channel               6494 non-null   str    
 3   transaction_type      6484 non-null   str    
 4   merchant              6483 non-null   str    
 5   city                  6483 non-null   str    
 6   device_type           6493 non-null   str    
 7   bank                  6490 non-null   str    
 8   failed_attempts       6500 non-null   int64  
 9   transaction_velocity  6500 non-null   float64
 10  international_flag    6500 non-null   int64  
 11  account_age_years     6500 non-null   int64  
 12  location_type         6497 non-null   str    
 13  fraud                 6500 non-null   int64  
dtypes: float64(2), int64(5), str(7)
memory usage: 711.1 KB


SEPERATE THE X AND Y

In [25]:
x = df.drop("fraud" , axis = 1)
y = df["fraud"]

NOW WE WILL HANDLE MISSING VALUES , CAT_COLUMNS AND IMPORT LIBRARIES FOR MISSING VALUES THEN USE COLUMN TRANSFORMER AND PIPELINE

In [26]:
from sklearn.impute import SimpleImputer
cat_pipeline = Pipeline([
    ("imputer" , SimpleImputer(strategy='most_frequent')),
    ("encoder" , OneHotEncoder(handle_unknown='ignore'))
])

In [27]:
preprocessor = ColumnTransformer(transformers=[
    ("num_imputer" , SimpleImputer(strategy='mean'),
     ["amount"]
     ),
    ("cat" , cat_pipeline , [
        "channel",
        "transaction_type",
        "merchant",
        "city",
        "device_type",
        "bank",
        "location_type"
    ])
])

In [28]:
pipeline = Pipeline([
    ("preprocessor" , preprocessor),
    ("classifier" , XGBClassifier(learning_rate=0.2))
])

FIRST WE WILL DO IT WITH CROSS VALIDATGION

In [29]:
print(df["fraud"].unique())
print(df["fraud"].value_counts(dropna=False))
print(df["fraud"].dtype)

[0 1]
fraud
0    5958
1     542
Name: count, dtype: int64
int64


In [30]:
x_train , x_test , y_train , y_test = train_test_split(x , y , test_size = 0.2 , random_state = 42 , stratify = y)

NOW Make Stratified

In [31]:
cv = StratifiedKFold(
    n_splits = 5,
    random_state = 42,
    shuffle = True
)

NOW CROSS VALIDATION

In [32]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(
    pipeline,
    x_train,
    y_train,
    cv = cv,
    scoring = 'precision'
)

In [37]:
print(scores)

[0.9125     0.91346154 0.90769231 0.91057692 0.91346154]


NOW TRAIN ON PIPELINE

In [38]:
pipeline.fit(x_train , y_train)
y_pred = pipeline.predict(x_test)

NOW PRINT CLASSIFICATION REPORT AND ACCURACY

In [41]:
from sklearn.metrics import classification_report , accuracy_score
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      1.00      0.96      1192
           1       0.20      0.01      0.02       108

    accuracy                           0.91      1300
   macro avg       0.56      0.50      0.49      1300
weighted avg       0.86      0.91      0.88      1300



In [42]:
print(accuracy_score(y_test, y_pred))

0.9146153846153846


NOW SAVE THE MODEL

In [43]:
import pickle
with open("fraud_detection_model.pkl" , 'wb') as file:
    pickle.dump(pipeline , file)